In [20]:
%%html
<style>
/*overwrite hard coded write background by vscode for ipywidges */
.cell-output-ipywidget-background {
   background-color: transparent !important;
}

/*set widget foreground text and color of interactive widget to vs dark theme color */
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>
<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>


In [ ]:
from brainhack import Tukey, Sequence, System, Simulator, CompositeDictionary, Signal
import matplotlib.pyplot as plt
import numpy
from pandas import read_pickle

from plotly.offline import init_notebook_mode
import plotly.graph_objects as go
from ipywidgets import widgets

init_notebook_mode()

In [ ]:
seqs = read_pickle(f"./sequenceParametrizations.pkl")

data = []
for i, row in seqs.iterrows():
    pulse = Tukey(
        duration=row['pw'] * 1e-3,
        shape=.2,
        flipAngle=row['FA'],
        offset=7e3,
    )
    sequence = Sequence(
        Signal.ihMTR_CM,
        pulse=pulse,
        N_pulsePerOffset=1,
        N_pulse=row['NP'],
        N_burst=row['NB'],
        N_adc=row['turbo'] - 3,
        N_dummyADC=3,
        dt_interPulse=row['dt'] * 1e-3,
        TR_burst=row['t_boost'] * 1e-3,
        dt_lastBurst=row['t_last_boost'] * 1e-3,
        es=row['es'] * 1e-3,
        tr=row['TR'] * 1e-3,
        readout_flipAngle=row['FA_readout'],
    )
    system = System(
        pulse=pulse,
        poolFree_M0=1 - .1565,
        poolFree_T1=2000 * 1e-3,
        poolFree_T2=25 * 1e-3,
        poolFreeBound_exchangeRate=11,
        poolBound_M0=.1565,
        poolBound_T1=500 * 1e-3,
        poolBound_T2=1/94696,
        poolBound_T1D=2.0601 * 1e-3,
        poolBound_lineshapeAsymmetry=-593
    )
    simulator = Simulator(system=system, sequence=sequence, output_vectorSlice=slice(1), export_readMatrix=False)
    data.append(simulator.SteadyState())

M0 = [dat['MT0'] for dat in data]
MTs = [.5 * (dat['MTs_Negative'] + dat['MTs_Positive']) for dat in data]
MTd_CM = [dat['MTd_CM'] for dat in data]

signals = CompositeDictionary({
    'MT0': M0,
    'MTs': MTs,
    'MTd_CM': MTd_CM
})

seqs['ihMT_CM'] = signals[Signal.ihMT_CM]
seqs['ihMTR_CM'] = signals[Signal.ihMTR_CM]

seqs['ihMT_CM'] = seqs['ihMT_CM'].mask(seqs.eval("TR > 1600"), 0)
seqs['ihMTR_CM'] = seqs['ihMTR_CM'].mask(seqs.eval("TR > 1600"), 0)

seqs['SNR_CM/sqrt(TR)'] = seqs['ihMT_CM'] / numpy.sqrt(seqs['TR'])


In [4]:
subseqs = seqs[['FA', 'pw', 'NP', 'NB', 't_boost', 'TR', 'SNR_CM/sqrt(TR)', 'ihMTR_CM', 'cap_antenna']]

cap_antennas = numpy.unique(subseqs['cap_antenna'])
pws = numpy.unique(subseqs['pw'])
NPs = numpy.unique(subseqs['NP'])
NBs = numpy.unique(subseqs['NB'])
BTRs = numpy.unique(subseqs['t_boost'])

maxihMTR = numpy.max(subseqs['ihMTR_CM'])
maxSNRsqrtTR = numpy.max(subseqs['SNR_CM/sqrt(TR)'])
minBTR = numpy.min(subseqs['t_boost'])
maxBTR = numpy.max(subseqs['t_boost'])

In [42]:
caps = {numpy.round(cap_antenna * numpy.sqrt(2)).astype(int): cap_antenna for cap_antenna in cap_antennas}
cap = widgets.SelectionSlider(
    options=[str(cap) + ' µT' for cap in caps.keys()],
    description="B1Peak",
    readout=True,
)
pw = widgets.SelectionSlider(
    options=[str(pw) + ' ms' for pw in pws],
    description="pw",
    readout=True,
)
np = widgets.SelectionSlider(
    options=[str(np) for np in NPs],
    description="NP",
    readout=True,
)

box = widgets.HBox([cap, np, pw])
tracesSNR = [go.Scatter(x=BTRs, y=[0] * len(BTRs), name=f'NB = {i}', hovertemplate='TR = %{customdata[0]} ms') for i in NBs]
tracesihMTR = [go.Scatter(x=BTRs, y=[0] * len(BTRs), name=f'NB = {i}', hovertemplate='TR = %{customdata[0]} ms') for i in NBs]

g1 = go.FigureWidget(
    data=tracesSNR,
    layout=go.Layout(
        title=dict(
          text= r'$\mathrm{FA}_\mathrm{sat} = \mathrm{-}°$',
          x=.5
        ),
        hovermode='x unified',
        template='plotly_dark',
        yaxis=dict(
             title=r'$\mathrm{SNR} \div \sqrt{\mathrm{TR}} \quad \mathrm{[a.u.]}$',
             range=[0, maxSNRsqrtTR]
        ),
        xaxis=dict(
             title=r'$\mathrm{BTR} \quad \mathrm{[ms]}$'
        )
    )
)

g2 = go.FigureWidget(
    data=tracesihMTR,
    layout=go.Layout(
        title=dict(
          text=r'$\mathrm{FA}_\mathrm{sat} = \mathrm{-}°$',
          x=.5
        ),
        hovermode='x unified',
        template='plotly_dark',
        yaxis=dict(
             title=r'$\mathrm{ihMTR} \quad \mathrm{[\%]}$',
             range=[0, maxihMTR]
        ),
        xaxis=dict(
             title=r'$\mathrm{BTR} \quad \mathrm{[ms]}$'
        )
    )
)

def response(*args, **kwargs):
        filtered = subseqs[(subseqs.NP == int(np.value)) & (subseqs.pw == float(pw.value.replace(' ms', ''))) & (subseqs.cap_antenna == caps[int(cap.value.replace(' µT', ''))])]
        with (g1.batch_update(), g2.batch_update()):
            for i, nb in enumerate(NBs):
                g1.data[i].y = filtered[(subseqs.NB == nb)]['SNR_CM/sqrt(TR)'].tolist()  # tolist() to avoid plotly from throwing an error from "if not value: ..." when value is NDArray
                g1.data[i].customdata = numpy.stack([filtered[(subseqs.NB == nb)]['TR'].tolist()], axis=-1)
                g2.data[i].y = filtered[(subseqs.NB == nb)]['ihMTR_CM'].tolist()  # tolist() to avoid plotly from throwing an error from "if not value: ..." when value is NDArray
                g2.data[i].customdata = numpy.stack([filtered[(subseqs.NB == nb)]['TR'].tolist()], axis=-1)
            g1.layout.title.text = r'$\mathrm{FA}_\mathrm{sat} = ' + f'{int(filtered['FA'].iloc[0])}°$'
            g2.layout.title.text = r'$\mathrm{FA}_\mathrm{sat} = ' + f'{int(filtered['FA'].iloc[0])}°$'

cap.observe(response, names="value")
pw.observe(response, names="value")
np.observe(response, names="value")

widgets.VBox([box, widgets.HBox([g1, g2])])